In [ ]:
# !pip install xgboost lightgbm optuna

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 40.1 MB/s  0:00:00

   ---------- ----------------------------- 2/8 [greenlet]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   -------------------- ------------------- 4/8 [sqlalchemy]
   ------------------------------ --------- 6/8 [alembic]
   ------------------------------ --------- 6/8 [alembic]
   ----------------------------------- ---- 7/8 [optuna]
   ----------------------------------- ---- 7/8 [optuna]
   -------

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score,
    precision_score,
    f1_score,
    confusion_matrix,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import optuna

print('라이브러리 불러오기 완료')

라이브러리 불러오기 완료


In [3]:
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'Malgun Gothic'

In [5]:
data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)

df['target_original'] = data.target

In [7]:
df['target'] = 1 - df['target_original']

df['target_name'] = df['target'].map({
    0: 'benign',
    1: 'malignant'
})

In [8]:
print('데이터 크기 (행, 열):', df.shape)

print('\n원본 target 기준 개수')
print(df['target_original'].value_counts().sort_index())

print('\n수업용 target 기준 개수:')
print(df['target'].value_counts().sort_index())

print('\n수업용 target 이름 기준 개수:')
print(df['target_name'].value_counts())

데이터 크기 (행, 열): (569, 33)

원본 target 기준 개수
target_original
0    212
1    357
Name: count, dtype: int64

수업용 target 기준 개수:
target
0    357
1    212
Name: count, dtype: int64

수업용 target 이름 기준 개수:
target_name
benign       357
malignant    212
Name: count, dtype: int64


In [9]:
X = df.drop(columns=['target_original', 'target', 'target_name'])

y = df['target']

print('X 크기:', X.shape)
print('y 크기:', y.shape)

X 크기: (569, 30)
y 크기: (569,)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('train 크기:', X_train.shape)
print('test 크기:', X_test.shape)
print('\ntrain의 target 비율:')
print(y_train.value_counts(normalize=True).round(3))
print('\ntest의 target 비율:')
print(y_test.value_counts(normalize=True).round(3))

train 크기: (455, 30)
test 크기: (114, 30)

train의 target 비율:
target
0    0.626
1    0.374
Name: proportion, dtype: float64

test의 target 비율:
target
0    0.632
1    0.368
Name: proportion, dtype: float64


In [14]:
def evaluate_classification_model(model_name, y_true, y_pred, y_proba):
    
    accuracy = accuracy_score(y_true, y_pred)
    
    precision_malignant = precision_score(y_true, y_pred, pos_label=1)
    
    recall_malignant = recall_score(y_true, y_pred, pos_label=1)
    
    f1_malignant = f1_score(y_true, y_pred, pos_label=1)
    
    roc_auc = roc_auc_score(y_true, y_proba)
    
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    result = {
        'model_name': model_name,
        'accuracy': accuracy,
        'precision_malignant': precision_malignant,
        'recall_malignant': recall_malignant,
        'f1_malignant': f1_malignant,
        'roc_auc': roc_auc,
        'TN': tn,
        'FP': fp,
        'FN': fn,
        'TP': tp
    }
    
    return result

print('평가 함수 준비 완료')

평가 함수 준비 완료


In [15]:
xgb_base_model = XGBClassifier(
    n_estimators = 100,
    max_depth=3,
    learning_rate=0.1,
    random_state=42,
    eval_metrics='logloss'
)

In [16]:
xgb_base_model.fit(X_train, y_train)

xgb_base_pred = xgb_base_model.predict(X_test)

xgb_base_proba = xgb_base_model.predict_proba(X_test)[:, 1]

c:\Users\user\AppData\Local\anaconda3\envs\ml_env01\Lib\site-packages\xgboost\training.py:200: UserWarning: [17:27:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "eval_metrics" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [17]:
xgb_base_result = evaluate_classification_model(
    model_name='XGBoost Base',
    y_true=y_test,
    y_pred=xgb_base_pred,
    y_proba=xgb_base_proba
)

xgb_base_result

{'model_name': 'XGBoost Base',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9047619047619048,
 'f1_malignant': 0.95,
 'roc_auc': 0.9966931216931217,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(4),
 'TP': np.int64(38)}

In [18]:
print('=== Classification Report (XGBoost Base) ===')
print(classification_report(
    y_test,
    xgb_base_pred,
    target_names=['benign(0)', 'malignant(1)']
))

print('=== Confusion Matrix (XGBoost Base) ===')
print(confusion_matrix(y_test, xgb_base_pred))
print('\n[[TN, FP],')
print(' [FN, TP]] 순서입니다.')

=== Classification Report (XGBoost Base) ===
              precision    recall  f1-score   support

   benign(0)       0.95      1.00      0.97        72
malignant(1)       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.95      0.96       114
weighted avg       0.97      0.96      0.96       114

=== Confusion Matrix (XGBoost Base) ===
[[72  0]
 [ 4 38]]

[[TN, FP],
 [FN, TP]] 순서입니다.


In [19]:
xgb_for_random_search = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)

In [20]:
param_distributions = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [2, 3, 4, 5], 
    'learning_rate': [0.001, 0.003, 0.005, 0.1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1.0],
    'reg_lambda': [0.5, 1.0, 1.5, 2.0]
}

In [21]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print('탐색 범위 설정 완료')

탐색 범위 설정 완료


In [22]:
random_search = RandomizedSearchCV(
    estimator=xgb_for_random_search,
    param_distributions=param_distributions,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print('RandomizedSearchCV 탐색 완료')

RandomizedSearchCV 탐색 완료


In [23]:
print('Best ROC-AUC:', random_search.best_score_)
print('Best Params:', random_search.best_params_)
print('Best Estimator:', random_search.best_estimator_)

Best ROC-AUC: 0.9908565272831202
Best Params: {'subsample': 0.7, 'reg_lambda': 1.0, 'reg_alpha': 1.0, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.9}
Best Estimator: XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)


In [24]:
xgb_random_search_model = random_search.best_estimator_

xgb_random_search_pred = xgb_random_search_model.predict(X_test)
xgb_random_search_proba = xgb_random_search_model.predict_proba(X_test)[:, 1]

xgb_random_search_result = evaluate_classification_model(
    model_name='XGBoost RandomizedSearchCV',
    y_true=y_test,
    y_pred=xgb_random_search_pred,
    y_proba=xgb_random_search_proba
)

xgb_random_search_result

{'model_name': 'XGBoost RandomizedSearchCV',
 'accuracy': 0.9824561403508771,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9523809523809523,
 'f1_malignant': 0.975609756097561,
 'roc_auc': 0.9930555555555555,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(2),
 'TP': np.int64(40)}

In [28]:
def objective(trial):
    
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 3.0),
        'random_state': 42,
        'eval_metric': 'logloss'
    }
    
    model = XGBClassifier(**params)
    
    cv = StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )
    
    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )
    
    return scores.mean()

print('objective 함수 준비 완료')

objective 함수 준비 완료


In [30]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction='maximize',
    sampler=sampler
)

study.optimize(objective, n_trials=20)

print('Optuna 튜닝 완료')

Optuna 튜닝 완료


In [31]:
print('Best ROC-AUC:', study.best_value)
print('Best Params:', study.best_params)

study.trials_dataframe().head()

Best ROC-AUC: 0.9911082530888625
Best Params: {'n_estimators': 294, 'max_depth': 3, 'learning_rate': 0.06729392009628148, 'subsample': 0.77457520756364, 'colsample_bytree': 0.909182130134382, 'reg_alpha': 0.07977803963614018, 'reg_lambda': 2.659229016644547}


,number,value,datetime_start,datetime_complete,duration,params_colsample_bytree,params_learning_rate,params_max_depth,params_n_estimators,params_reg_alpha,params_reg_lambda,params_subsample,state
0,0,0.990299,2026-06-18 21:25:12.115625,2026-06-18 21:25:12.509395,0 days 00:00:00.393770,0.746806,0.089608,6,144,0.155995,0.645209,0.879598,COMPLETE
1,1,0.990432,2026-06-18 21:25:12.509395,2026-06-18 21:25:12.937308,0 days 00:00:00.427913,0.990973,0.083411,5,267,0.832443,1.030848,0.706175,COMPLETE
2,2,0.986328,2026-06-18 21:25:12.937308,2026-06-18 21:25:13.054211,0 days 00:00:00.116903,0.829584,0.024879,2,95,0.291229,2.029632,0.857427,COMPLETE
3,3,0.987573,2026-06-18 21:25:13.054211,2026-06-18 21:25:13.183372,0 days 00:00:00.129161,0.935553,0.029967,3,85,0.199674,1.785586,0.836821,COMPLETE
4,4,0.990421,2026-06-18 21:25:13.183372,2026-06-18 21:25:13.385601,0 days 00:00:00.202229,0.719515,0.061721,2,198,0.948886,2.914080,0.751157,COMPLETE


In [34]:
xgb_optuna_model = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric='logloss'
)

xgb_optuna_model.fit(X_train, y_train)

xgb_optuna_pred = xgb_optuna_model.predict(X_test)
xgb_optuna_proba = xgb_optuna_model.predict_proba(X_test)[:, 1]

xgb_optuna_result = evaluate_classification_model(
    model_name='XGBoost Optuna',
    y_true=y_test,
    y_pred=xgb_optuna_pred,
    y_proba=xgb_optuna_proba
)
xgb_optuna_result

{'model_name': 'XGBoost Optuna',
 'accuracy': 0.9736842105263158,
 'precision_malignant': 1.0,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9629629629629629,
 'roc_auc': 0.9923941798941799,
 'TN': np.int64(72),
 'FP': np.int64(0),
 'FN': np.int64(3),
 'TP': np.int64(39)}

In [35]:
xgb_tuning_results_df = pd.DataFrame([
    xgb_base_result,
    xgb_random_search_result,
    xgb_optuna_result
])

xgb_tuning_results_df

,model_name,accuracy,precision_malignant,recall_malignant,f1_malignant,roc_auc,TN,FP,FN,TP
0,XGBoost Base,0.964912,1.0,0.904762,0.950000,0.996693,72,0,4,38
1,XGBoost RandomizedSearchCV,0.982456,1.0,0.952381,0.975610,0.993056,72,0,2,40
2,XGBoost Optuna,0.973684,1.0,0.928571,0.962963,0.992394,72,0,3,39


In [37]:
logistic_model = Pipeline(
    steps=[
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000, random_state=42))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]

logistic_result = evaluate_classification_model(
    model_name='Logistic Regression',
    y_true=y_test,
    y_pred=logistic_pred,
    y_proba=logistic_proba
)
logistic_result

{'model_name': 'Logistic Regression',
 'accuracy': 0.9649122807017544,
 'precision_malignant': 0.975,
 'recall_malignant': 0.9285714285714286,
 'f1_malignant': 0.9512195121951219,
 'roc_auc': 0.996031746031746,
 'TN': np.int64(71),
 'FP': np.int64(1),
 'FN': np.int64(3),
 'TP': np.int64(39)}